# 02 - Traitement des Données Texte

## Section : Gestion du Texte

## Objectif
Transformer les données brutes en un dataset prêt pour la modélisation en :
- Nettoyant le HTML des descriptions
- Gérant les valeurs manquantes (35% de descriptions vides)
- Normalisant les textes
- Créant des features supplémentaires
- Préparant le dataset final pour la modélisation

## Plan
1. Chargement des données
2. Phase 1 : Nettoyage HTML
3. Phase 2 : Gestion des valeurs manquantes
4. Phase 3 : Normalisation des textes
5. Phase 4 : Feature Engineering
6. Phase 5 : Préparation finale et sauvegarde
   - 6.2 bis. Superclasse Publications (optionnel, à tester)



In [1]:
# Import des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Racine du projet : chercher le dossier contenant 'src'
current = Path.cwd()
PROJECT_ROOT = current
while not (PROJECT_ROOT / 'src').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        PROJECT_ROOT = current  # fallback
        break
    PROJECT_ROOT = parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import des modules de preprocessing
from src.utils.data_loader import load_data, save_data
from src.preprocessing import (
    clean_html,
    decode_html_entities,
    has_html,
    remove_remaining_html,
    normalize_text,
    combine_texts,
    create_length_features,
    create_binary_features,
    create_quality_features,
    PreprocessingPipeline
)

# Configuration des graphiques
try:
    plt.style.use('seaborn-v0_8')
except (OSError, ValueError):
    plt.style.use('ggplot')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Bibliothèques importées avec succès")



✅ Bibliothèques importées avec succès


## 1. Chargement des Données

Chargement des données brutes depuis le dossier `data brut/`.


In [2]:
# Chargement des données
DATA_DIR = PROJECT_ROOT / 'data brut'

X_train, y_train, X_test = load_data(data_dir=DATA_DIR)

# Vérification de l'alignement
if X_train.shape[0] == y_train.shape[0]:
    print(f"\n✅ Alignement OK : X_train et y_train ont le même nombre de lignes")
else:
    print(f"\n⚠️  ATTENTION : X_train et y_train n'ont pas le même nombre de lignes !")

# Aperçu des données
print("\n" + "="*80)
print("APERÇU DE X_train (avant preprocessing)")
print("="*80)
print(X_train.head())
print(f"\nValeurs manquantes :")
print(X_train.isnull().sum())



🔄 Chargement des données...
✅ Données chargées avec succès !
  - X_train : 84,916 lignes × 5 colonnes
  - y_train : 84,916 lignes × 2 colonnes
  - X_test  : 13,812 lignes × 5 colonnes

✅ Alignement OK : X_train et y_train ont le même nombre de lignes

APERÇU DE X_train (avant preprocessing)
   Unnamed: 0                                        designation  \
0           0  Olivia: Personalisiertes Notizbuch / 150 Seite...   
1           1  Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...   
2           2  Grand Stylet Ergonomique Bleu Gamepad Nintendo...   
3           3  Peluche Donald - Europe - Disneyland 2000 (Mar...   
4           4                               La Guerre Des Tuques   

                                         description   productid     imageid  
0                                                NaN  3804725264  1263597046  
1                                                NaN   436067568  1008141237  
2  PILOT STYLE Touch Pen de marque Speedlink est ...   2011151

## 2. Phase 1 : Nettoyage HTML

### 2.1 Analyse du HTML avant nettoyage


In [3]:
# Détection du HTML dans les descriptions
X_train['has_html_before'] = X_train['description'].apply(has_html)
X_test['has_html_before'] = X_test['description'].apply(has_html)

html_count_train = X_train['has_html_before'].sum()
html_count_test = X_test['has_html_before'].sum()

print("📊 Présence de HTML dans les descriptions (AVANT nettoyage) :")
print(f"  - X_train : {html_count_train} descriptions avec HTML ({html_count_train/len(X_train)*100:.2f}%)")
print(f"  - X_test : {html_count_test} descriptions avec HTML ({html_count_test/len(X_test)*100:.2f}%)")

# Exemples de descriptions avec HTML (textes courts, affichage intégral)
print("\n📝 Exemples de descriptions avec HTML (textes courts, affichage intégral) :")
html_mask = X_train['has_html_before']
short_mask = X_train['description'].fillna('').str.len() <= 300
html_examples = X_train.loc[html_mask & short_mask, 'description'].head(3)
for idx, desc in enumerate(html_examples, 1):
    print(f"\nExemple {idx} :")
    print(desc)



📊 Présence de HTML dans les descriptions (AVANT nettoyage) :
  - X_train : 15645 descriptions avec HTML (18.42%)
  - X_test : 2478 descriptions avec HTML (17.94%)

📝 Exemples de descriptions avec HTML (textes courts, affichage intégral) :

Exemple 1 :
Nom de la marque:NoEnName_Null<br />Affichage de couleur:Oui<br />Ecran tactile:Pas de<br />Numéro du modèle:CoolBaby<br />Paquet:Oui<br />Communication:USB

Exemple 2 :
<br>Attention !!! Ce produit est un import  si les informations 'langues' et 'sous-titres' n'apparaissent pas sur cette fiche produit c'est que l'éditeur ne nous les a pas fournies. Néanmoins dans la grande majorité de ces cas il n'existe ni langue ni sous titres en français sur ces imports.

Exemple 3 :
<br>Attention !!! Ce produit est un import  si les informations 'langues' et 'sous-titres' n'apparaissent pas sur cette fiche produit c'est que l'éditeur ne nous les a pas fournies. Néanmoins dans la grande majorité de ces cas il n'existe ni langue ni sous titres en franç

### 2.2 Nettoyage du HTML


In [4]:
# Nettoyage HTML des descriptions
print("🔄 Nettoyage du HTML...")

X_train['description_clean'] = X_train['description'].apply(
    lambda x: clean_html(x, preserve_structure=True)
)
X_test['description_clean'] = X_test['description'].apply(
    lambda x: clean_html(x, preserve_structure=True)
)

# Vérification après nettoyage
X_train['has_html_after'] = X_train['description_clean'].apply(has_html)
X_test['has_html_after'] = X_test['description_clean'].apply(has_html)

html_remaining_train = X_train['has_html_after'].sum()
html_remaining_test = X_test['has_html_after'].sum()

print(f"✅ Nettoyage terminé !")
print(f"  - X_train : {html_remaining_train} descriptions avec HTML restant ({html_remaining_train/len(X_train)*100:.2f}%)")
print(f"  - X_test : {html_remaining_test} descriptions avec HTML restant ({html_remaining_test/len(X_test)*100:.2f}%)")

# Comparaison longueurs avant/après
X_train['desc_length_before'] = X_train['description'].fillna('').astype(str).str.len()
X_train['desc_length_after'] = X_train['description_clean'].str.len()

print(f"\n📊 Comparaison des longueurs (X_train) :")
print(f"  - Avant nettoyage : moyenne = {X_train['desc_length_before'].mean():.1f}, médiane = {X_train['desc_length_before'].median():.1f}")
print(f"  - Après nettoyage : moyenne = {X_train['desc_length_after'].mean():.1f}, médiane = {X_train['desc_length_after'].median():.1f}")

# Exemples après nettoyage (textes courts, affichage intégral)
print("\n📝 Exemples de descriptions nettoyées (textes courts, affichage intégral) :")
html_mask = X_train['has_html_before']
short_mask = X_train['description'].fillna('').str.len() <= 300
subset = X_train.loc[html_mask & short_mask, ['description', 'description_clean']].head(3)
for i, (before, after) in enumerate(zip(subset['description'], subset['description_clean']), 1):
    print(f"\nExemple {i} :")
    print(f"  AVANT : {before}")
    print(f"  APRÈS : {after}")



🔄 Nettoyage du HTML...
✅ Nettoyage terminé !
  - X_train : 29 descriptions avec HTML restant (0.03%)
  - X_test : 6 descriptions avec HTML restant (0.04%)

📊 Comparaison des longueurs (X_train) :
  - Avant nettoyage : moyenne = 524.6, médiane = 231.0
  - Après nettoyage : moyenne = 488.8, médiane = 220.0

📝 Exemples de descriptions nettoyées (textes courts, affichage intégral) :

Exemple 1 :
  AVANT : Nom de la marque:NoEnName_Null<br />Affichage de couleur:Oui<br />Ecran tactile:Pas de<br />Numéro du modèle:CoolBaby<br />Paquet:Oui<br />Communication:USB
  APRÈS : Nom de la marque:NoEnName_Null Affichage de couleur:Oui Ecran tactile:Pas de Numéro du modèle:CoolBaby Paquet:Oui Communication:USB

Exemple 2 :
  AVANT : <br>Attention !!! Ce produit est un import  si les informations 'langues' et 'sous-titres' n'apparaissent pas sur cette fiche produit c'est que l'éditeur ne nous les a pas fournies. Néanmoins dans la grande majorité de ces cas il n'existe ni langue ni sous titres en frança

## 3. Phase 2 : Gestion des Valeurs Manquantes

### 3.1 Analyse des valeurs manquantes


In [5]:
# Analyse des valeurs manquantes
missing_train = X_train['description_clean'].isna() | (X_train['description_clean'] == '')
missing_test = X_test['description_clean'].isna() | (X_test['description_clean'] == '')

missing_count_train = missing_train.sum()
missing_count_test = missing_test.sum()

print("📊 Analyse des valeurs manquantes (descriptions) :")
print(f"  - X_train : {missing_count_train} descriptions vides ({missing_count_train/len(X_train)*100:.2f}%)")
print(f"  - X_test : {missing_count_test} descriptions vides ({missing_count_test/len(X_test)*100:.2f}%)")

# Analyse par classe (si y_train disponible)
if 'prdtypecode' not in X_train.columns:
    # Fusionner avec y_train pour analyser par classe
    df_train_merged = pd.concat([X_train, y_train], axis=1)
else:
    df_train_merged = X_train.copy()

if 'prdtypecode' in df_train_merged.columns:
    missing_by_class = df_train_merged.groupby('prdtypecode').agg({
        'description_clean': lambda x: (x.isna() | (x == '')).sum() / len(x) * 100
    }).round(2).sort_values('description_clean', ascending=False)
    
    print(f"\n📊 Top 10 classes avec le plus de descriptions manquantes :")
    print(missing_by_class.head(10))



📊 Analyse des valeurs manquantes (descriptions) :
  - X_train : 29888 descriptions vides (35.20%)
  - X_test : 4903 descriptions vides (35.50%)

📊 Top 10 classes avec le plus de descriptions manquantes :
             description_clean
prdtypecode                   
2403                     97.36
2462                     96.27
2280                     93.28
1160                     91.17
10                       89.15
1180                     79.97
40                       65.47
1140                     64.62
2705                     37.12
1320                     33.97


### 3.2 Application de la stratégie de gestion des valeurs manquantes

**Stratégie choisie :** `designation_only` - Pour les produits sans description, on utilisera uniquement la designation.


In [6]:
# Les descriptions vides sont déjà gérées (chaîne vide)
# On s'assure qu'elles sont bien des chaînes vides et non NaN
X_train['description_clean'] = X_train['description_clean'].fillna('')
X_test['description_clean'] = X_test['description_clean'].fillna('')

print("✅ Valeurs manquantes gérées : descriptions vides remplacées par chaîne vide")
print(f"  - X_train : {X_train['description_clean'].isna().sum()} NaN restants")
print(f"  - X_test : {X_test['description_clean'].isna().sum()} NaN restants")



✅ Valeurs manquantes gérées : descriptions vides remplacées par chaîne vide
  - X_train : 0 NaN restants
  - X_test : 0 NaN restants


## 4. Phase 3 : Normalisation des Textes

Normalisation des désignations et descriptions nettoyées.


In [7]:
# Normalisation des textes
print("🔄 Normalisation des textes...")

# Normalisation des désignations
X_train['designation_clean'] = X_train['designation'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)
X_test['designation_clean'] = X_test['designation'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)

# Normalisation des descriptions (déjà nettoyées du HTML)
X_train['description_clean'] = X_train['description_clean'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)
X_test['description_clean'] = X_test['description_clean'].apply(
    lambda x: normalize_text(x, lowercase=True, remove_extra_spaces=True, remove_accents=False)
)

# Passe de sécurité : supprimer tout HTML restant (malformé : < li>, <br/>, <. 4pc br>, etc.)
X_train['description_clean'] = X_train['description_clean'].apply(remove_remaining_html)
X_test['description_clean'] = X_test['description_clean'].apply(remove_remaining_html)

print("✅ Normalisation terminée !")

# Exemples
print("\n📝 Exemples de textes normalisés :")
for idx in range(3):
    print(f"\nExemple {idx+1} :")
    print(f"  Designation originale : {X_train['designation'].iloc[idx][:100]}...")
    print(f"  Designation normalisée : {X_train['designation_clean'].iloc[idx][:100]}...")



🔄 Normalisation des textes...
✅ Normalisation terminée !

📝 Exemples de textes normalisés :

Exemple 1 :
  Designation originale : Olivia: Personalisiertes Notizbuch / 150 Seiten / Punktraster / Ca Din A5 / Rosen-Design...
  Designation normalisée : olivia: personalisiertes notizbuch / 150 seiten / punktraster / ca din a5 / rosen-design...

Exemple 2 :
  Designation originale : Journal Des Arts (Le) N° 133 Du 28/09/2001 - L'art Et Son Marche Salon D'art Asiatique A Paris - Jac...
  Designation normalisée : journal des arts (le) n° 133 du 28/09/2001 - l'art et son marche salon d'art asiatique a paris - jac...

Exemple 3 :
  Designation originale : Grand Stylet Ergonomique Bleu Gamepad Nintendo Wii U - Speedlink Pilot Style...
  Designation normalisée : grand stylet ergonomique bleu gamepad nintendo wii u - speedlink pilot style...


## 5. Phase 4 : Feature Engineering

Création de features supplémentaires pour améliorer la classification.


In [8]:
# Création de toutes les features
print("🔄 Création des features...")

# Utiliser les colonnes originales pour les features de longueur
X_train_features = X_train[['designation', 'description']].copy()
X_test_features = X_test[['designation', 'description']].copy()

# Créer les features
X_train_features = create_length_features(X_train_features)
X_train_features = create_binary_features(X_train_features)
X_train_features = create_quality_features(X_train_features)

X_test_features = create_length_features(X_test_features)
X_test_features = create_binary_features(X_test_features)
X_test_features = create_quality_features(X_test_features)

# Ajouter les features au DataFrame principal
feature_cols = [col for col in X_train_features.columns if col not in ['designation', 'description']]
for col in feature_cols:
    X_train[col] = X_train_features[col]
    X_test[col] = X_test_features[col]

print(f"✅ {len(feature_cols)} features créées :")
for col in feature_cols:
    print(f"  - {col}")

# Aperçu des features
print("\n📊 Aperçu des features créées :")
print(X_train[feature_cols].describe())



🔄 Création des features...
✅ 14 features créées :
  - designation_length
  - description_length
  - total_text_length
  - designation_word_count
  - description_word_count
  - total_word_count
  - designation_avg_word_length
  - description_avg_word_length
  - has_description
  - has_html
  - is_description_empty
  - text_completeness
  - description_quality_score
  - word_density

📊 Aperçu des features créées :
       designation_length  description_length  total_text_length  \
count        84916.000000        84916.000000       84916.000000   
mean            70.163303          524.555926         594.719228   
std             36.793383          754.893905         762.067621   
min             11.000000            0.000000          11.000000   
25%             43.000000            0.000000          65.000000   
50%             64.000000          231.000000         313.000000   
75%             90.000000          823.000000         903.000000   
max            250.000000        12451.0

## 6. Phase 5 : Préparation Finale

### 6.1 Combinaison des textes


In [9]:
# Combiner designation et description en un seul texte
print("🔄 Combinaison des textes...")

X_train['text_combined'] = X_train.apply(
    lambda row: combine_texts(
        row['designation_clean'],
        row['description_clean'],
        separator=' ',
        handle_missing='designation_only'
    ),
    axis=1
)

X_test['text_combined'] = X_test.apply(
    lambda row: combine_texts(
        row['designation_clean'],
        row['description_clean'],
        separator=' ',
        handle_missing='designation_only'
    ),
    axis=1
)

print("✅ Textes combinés !")

# Statistiques sur les textes combinés
print(f"\n📊 Statistiques sur les textes combinés (X_train) :")
print(f"  - Longueur moyenne : {X_train['text_combined'].str.len().mean():.1f} caractères")
print(f"  - Longueur médiane : {X_train['text_combined'].str.len().median():.1f} caractères")
print(f"  - Longueur min : {X_train['text_combined'].str.len().min()} caractères")
print(f"  - Longueur max : {X_train['text_combined'].str.len().max()} caractères")

# Vérifier qu'il n'y a pas de textes vides
empty_texts = (X_train['text_combined'] == '').sum()
print(f"  - Textes vides : {empty_texts} ({empty_texts/len(X_train)*100:.2f}%)")



🔄 Combinaison des textes...
✅ Textes combinés !

📊 Statistiques sur les textes combinés (X_train) :
  - Longueur moyenne : 559.2 caractères
  - Longueur médiane : 292.0 caractères
  - Longueur min : 11 caractères
  - Longueur max : 12212 caractères
  - Textes vides : 0 (0.00%)


### 6.2 Préparation des datasets finaux


In [10]:
# Sélectionner les colonnes importantes pour le dataset final
columns_to_keep = [
    'productid',
    'imageid',
    'designation_clean',
    'description_clean',
    'text_combined',
    'designation_length',
    'description_length',
    'total_text_length',
    'designation_word_count',
    'description_word_count',
    'total_word_count',
    'has_description',
    'has_html',
    'is_description_empty',
    'text_completeness',
    'description_quality_score',
    'word_density'
]

# Créer les datasets finaux
X_train_final = X_train[columns_to_keep].copy()
X_test_final = X_test[columns_to_keep].copy()

print("✅ Datasets finaux préparés !")
print(f"\n📊 Structure des datasets finaux :")
print(f"  - X_train_final : {X_train_final.shape[0]:,} lignes × {X_train_final.shape[1]} colonnes")
print(f"  - X_test_final : {X_test_final.shape[0]:,} lignes × {X_test_final.shape[1]} colonnes")
print(f"\nColonnes :")
for col in columns_to_keep:
    print(f"  - {col}")



✅ Datasets finaux préparés !

📊 Structure des datasets finaux :
  - X_train_final : 84,916 lignes × 17 colonnes
  - X_test_final : 13,812 lignes × 17 colonnes

Colonnes :
  - productid
  - imageid
  - designation_clean
  - description_clean
  - text_combined
  - designation_length
  - description_length
  - total_text_length
  - designation_word_count
  - description_word_count
  - total_word_count
  - has_description
  - has_html
  - is_description_empty
  - text_completeness
  - description_quality_score
  - word_density


### 6.3 Sauvegarde des datasets nettoyés


In [11]:
# Sauvegarder les datasets nettoyés
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("💾 Sauvegarde des datasets...")

# Sauvegarder les datasets finaux
save_data(X_train_final, OUTPUT_DIR / 'X_train_clean.csv')
save_data(X_test_final, OUTPUT_DIR / 'X_test_clean.csv')
save_data(y_train, OUTPUT_DIR / 'y_train.csv')

# Créer y_train_superclass si besoin (au cas où la cellule 6.2 bis n'a pas été exécutée)
if 'y_train_superclass' not in locals():
    PUBLICATION_CLASSES = {10, 2280, 2403, 2705}
    CODE_SUPERCLASSE = 9999
    y_train_superclass = y_train.copy()
    y_train_superclass['prdtypecode'] = y_train['prdtypecode'].apply(
        lambda x: CODE_SUPERCLASSE if x in PUBLICATION_CLASSES else x
    )

save_data(y_train_superclass, OUTPUT_DIR / 'y_train_superclass.csv')

print("\n✅ Datasets sauvegardés dans :", OUTPUT_DIR)
print("   - y_train.csv : 27 classes (référence)")
print("   - y_train_superclass.csv : 24 classes (Publications fusionnées, à tester)")



💾 Sauvegarde des datasets...
✅ Données sauvegardées : c:\Users\pruvost_t\Desktop\Cours\Datascientest_projets\data\processed\X_train_clean.csv
✅ Données sauvegardées : c:\Users\pruvost_t\Desktop\Cours\Datascientest_projets\data\processed\X_test_clean.csv
✅ Données sauvegardées : c:\Users\pruvost_t\Desktop\Cours\Datascientest_projets\data\processed\y_train.csv
✅ Données sauvegardées : c:\Users\pruvost_t\Desktop\Cours\Datascientest_projets\data\processed\y_train_superclass.csv

✅ Datasets sauvegardés dans : c:\Users\pruvost_t\Desktop\Cours\Datascientest_projets\data\processed
   - y_train.csv : 27 classes (référence)
   - y_train_superclass.csv : 24 classes (Publications fusionnées, à tester)


### 6.2 bis. Création de la superclasse Publications (à tester)

**⚠️ Justification :** Nous avons un doute sur la pertinence du regroupement des 4 classes 
"publications" (10-Livres, 2280-Presse, 2403-BD/Partitions, 2705-Romans). Ces classes partagent 
un vocabulaire très similaire et présentent des confusions importantes en modélisation. 
**Nous devons tester** si une fusion améliore les performances. Une version avec superclasse 
est donc créée en parallèle ; les modèles seront entraînés sur les deux versions pour comparaison.

In [12]:
# Classes à fusionner en superclasse "Publications" (10, 2280, 2403, 2705)
PUBLICATION_CLASSES = {10, 2280, 2403, 2705}
CODE_SUPERCLASSE = 9999  # Code unique pour la superclasse Publications

# Créer y_train avec superclasse (les 4 classes publications -> 9999)
y_train_superclass = y_train.copy()
y_train_superclass['prdtypecode'] = y_train['prdtypecode'].apply(
    lambda x: CODE_SUPERCLASSE if x in PUBLICATION_CLASSES else x
)

print("📊 Superclasse Publications créée :")
print(f"   - Classes fusionnées : {sorted(PUBLICATION_CLASSES)} (Livres, Presse, BD/Partitions, Romans)")
print(f"   - Nouveau code : {CODE_SUPERCLASSE}")
print(f"   - Avant : {y_train['prdtypecode'].nunique()} classes")
print(f"   - Après : {y_train_superclass['prdtypecode'].nunique()} classes")
print(f"   - Effectif superclasse : {(y_train_superclass['prdtypecode'] == CODE_SUPERCLASSE).sum():,} produits")

📊 Superclasse Publications créée :
   - Classes fusionnées : [10, 2280, 2403, 2705] (Livres, Presse, BD/Partitions, Romans)
   - Nouveau code : 9999
   - Avant : 27 classes
   - Après : 24 classes
   - Effectif superclasse : 15,411 produits
